# Câu 1. Tiền xử lý dữ liệu
**Input:** các file `*.txt` trong thư mục `../Data1`
**Output:** mỗi file sau tiền xử lý được lưu riêng (cùng tên) vào thư mục `../Data_Preprocessing`

Thứ tự thực hiện:
1. Chuẩn hóa văn bản (Standardizing Text) – sửa lỗi dấu thanh bị tách rời, chuẩn Unicode NFC
2. Chuyển văn bản thành chữ thường
3. Loại bỏ URL
4. Chuyển biểu tượng cảm xúc thành văn bản (Emojis and Emoticons)
5. Tách câu
6. Xóa dấu câu
7. Xóa chữ số
8. Sửa lỗi chính tả (Correcting Spelling)
9. Tách từ (Tokenizing)
10. Xóa các từ không có nghĩa (Stop Words)

*Lưu ý:* Tách câu và chuyển emoticon phải làm **trước** khi xóa dấu câu (vì cần dấu `.` `!` `?` để tách câu, và `:)` cũng là dấu câu). Xóa Stop Words làm **sau** khi tách từ vì bộ Stop Words có từ ghép (`bao_giờ`, `ai_nấy`...).

In [ ]:
# pip install underthesea emoji
import os
import re
import unicodedata
import emoji
from underthesea import word_tokenize

## Đọc dữ liệu

In [ ]:
path_in = "../Data1"                  # thư mục dữ liệu gốc
path_out = "../Data_Preprocessing"    # thư mục lưu kết quả
os.makedirs(path_out, exist_ok=True)

List_Files = sorted([f for f in os.listdir(path_in) if f.endswith(".txt")])
print("Số file dữ liệu:", len(List_Files))

# Xem thử 1 file
f = open(os.path.join(path_in, List_Files[0]), "r", encoding="utf-8")
print(f.read())
f.close()

## 1. Chuẩn hóa văn bản (Standardizing Text)
Một số file bị lỗi gõ dấu kiểu *Unicode tổ hợp*: dấu thanh nằm tách rời khỏi chữ (VD: `Uô ́ng ơ ̉đây` → `Uống ở đây`).
Nếu không sửa, máy sẽ hiểu `uô` + `́ng` là 2 từ khác nhau. Dùng `unicodedata.normalize("NFC", ...)` để ghép dấu thanh vào chữ.

In [ ]:
def text_std(text):
    text = re.sub(r"\s+([\u0300-\u0323])", r"\1", text)  # xóa khoảng trắng đứng trước dấu thanh rời
    text = unicodedata.normalize("NFC", text)               # ghép dấu thanh vào chữ (Unicode dựng sẵn)
    return text

print(text_std("Uô\u0301ng ơ\u0309 đây cu\u0303ng nhiê\u0300u lâ\u0300n"))

## 2. Chuyển văn bản thành chữ thường

In [ ]:
def to_lower(text):
    return text.lower()

print(to_lower("Chỉ có 2 chữ dành cho quán TỒI_TỆ"))

## 3. Loại bỏ URL

In [ ]:
def remove_url(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

print(remove_url("mình đặt qua https://www.foody.vn/da-nang/abc thấy ok, xem thêm www.facebook.com/xyz"))

## 4. Chuyển biểu tượng cảm xúc thành văn bản (Emojis and Emoticons)
- **Emoticon** (ghép bằng ký tự, VD `:)`, `:(`, `<3`): dùng từ điển `Dict_Emoticons`
- **Emoji** (VD 😍 ❤️ 👍): dùng thư viện `emoji`, đổi thành tên emoji (VD `red_heart`)

In [ ]:
Dict_Emoticons = {":))": " vui ", ":)": " vui ", "=))": " cười ", ":d": " cười ", "^^": " vui ",
                  ":((": " buồn ", ":(": " buồn ", ":'(": " khóc ", "<3": " yêu ", ":p": " tinh_nghịch ",
                  ";)": " nháy_mắt ", ":v": " cười ", ":3": " dễ_thương ", ":o": " ngạc_nhiên ", "-_-": " chán "}

def convert_emoji(text):
    # Emoticons
    for x in sorted(Dict_Emoticons, key=len, reverse=True):
        text = text.replace(x, Dict_Emoticons[x])
    # Emojis
    text = emoji.replace_emoji(text, replace=lambda chars, data: " " + data["en"].strip(":").replace("-", "_") + " ")
    return text

print(convert_emoji("quán đẹp 😍😍 đồ uống ngon ❤️ giá ok :) nhưng phục vụ chậm :("))

## 5. Tách câu
Tách theo dấu `.` `!` `?` và xuống dòng. Chỉ tách tại dấu chấm có khoảng trắng phía sau, để không cắt nhầm số như `50.000đ`.

In [ ]:
def split_sentences(text):
    List_Sentences = re.split(r"[.!?]+(?=\s|$)|\n+", text)
    return [s.strip() for s in List_Sentences if s.strip() != ""]

print(split_sentences("Giá 50.000đ/phần. Nhân viên vui vẻ! Sẽ quay lại?"))

## 6. Xóa dấu câu
Thay dấu câu bằng khoảng trắng; giữ lại dấu `_` của từ ghép (`nhân_viên`).

In [ ]:
def remove_punctuation(text):
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())

print(remove_punctuation('chỉ có 2 chữ dành cho quán " tồi_tệ " , bia 30k/thùng'))

## 7. Xóa chữ số
Xóa cả các từ có chứa số (VD `30k`, `8`, `000đ`) để không còn sót ký tự rác như `k`, `đ`.

In [ ]:
def remove_number(text):
    text = re.sub(r"\w*\d\w*", " ", text)
    return " ".join(text.split())

print(remove_number("bia phụ_thu 30k thùng giá 50 000đ phần 8 10 điểm"))

## 8. Sửa lỗi chính tả (Correcting Spelling)
TextBlob chỉ hỗ trợ tiếng Anh, nên với tiếng Việt dùng từ điển teencode/viết tắt (giống `lookup_dict` ở Ví dụ 3.6).

In [ ]:
Dict_TeenCode = {"ko": "không", "k": "không", "kh": "không", "khong": "không", "hok": "không", "hem": "không",
                 "đc": "được", "dc": "được", "đươc": "được", "nv": "nhân viên", "ng": "người", "mn": "mọi người",
                 "j": "gì", "vs": "với", "bt": "bình thường", "r": "rồi", "lm": "làm", "qá": "quá", "wá": "quá",
                 "ntn": "như thế nào", "thik": "thích", "iu": "yêu", "trc": "trước", "sp": "sản phẩm",
                 "a": "anh", "e": "em", "oke": "ok", "okie": "ok", "okela": "ok", "ngta": "người ta", "cx": "cũng"}

def correct_spelling(text):
    text = re.sub(r"(\w)\1{2,}", r"\1", text)   # chữ kéo dài: ngonnnn -> ngon, quáaaa -> quá
    words = text.split()
    text_pre = ""
    for word in words:
        if word in Dict_TeenCode:
            word = Dict_TeenCode[word]
        text_pre = text_pre + " " + word
    return text_pre.strip()

print(correct_spelling("ko ngonnnn như mong đợi nv phục vụ chậm hok quay lại"))

## 9. Tách từ (Tokenizing)
Dữ liệu gốc tách từ **không đồng nhất**: có file đã nối từ ghép bằng `_` (VD `trân_châu đường_đen`), có file chưa.
- Nếu đưa nguyên câu vào `underthesea`, nó dễ ghép nhầm với từ đã có `_` (VD `buồn nhân_viên` → `buồn_nhân_viên`).
- Nếu bỏ hết `_` rồi tách lại, có thể làm hỏng từ đã tách đúng (VD `trân_châu đường_đen` → `châu_đường`).

Cách làm: **giữ nguyên các từ đã có `_`** (gồm cả tên emoji như `red_heart`), chỉ đưa **các cụm từ liên tiếp chưa có `_`** vào `underthesea.word_tokenize(format="text")`.

In [ ]:
def tokenizing(text):
    List_Words = []
    chunk = []    # cụm từ liên tiếp chưa được tách
    for w in text.split():
        if "_" in w:
            if len(chunk) > 0:
                List_Words.append(word_tokenize(" ".join(chunk), format="text"))
                chunk = []
            List_Words.append(w)     # từ đã tách sẵn -> giữ nguyên
        else:
            chunk.append(w)
    if len(chunk) > 0:
        List_Words.append(word_tokenize(" ".join(chunk), format="text"))
    return " ".join(List_Words)

print(tokenizing("không ngon như mong đợi buồn nhân_viên phục vụ chậm red_heart"))
print(tokenizing("trà sữa trân_châu đường_đen ngon trân_châu dẻo"))

## 10. Xóa các từ không có nghĩa (Stop Words)
Dùng từ điển `vietnamese-stopwords.txt` (đặt cùng thư mục với notebook này). Từ ghép trong từ điển viết bằng khoảng trắng nên đổi sang `_` cho khớp với kết quả tách từ.

In [ ]:
f = open("vietnamese-stopwords.txt", "r", encoding="utf-8")
List_StopWords = [w.strip().replace(" ", "_") for w in f.read().split("\n") if w.strip() != ""]
f.close()
print("Số stop words:", len(List_StopWords))

def remove_stopwords(text):
    return " ".join(word for word in text.split() if word not in List_StopWords)

print(remove_stopwords("nhân_viên phục_vụ thì ăn_mặc lịch_sự lắm không bao_giờ quay lại"))

## Tiền xử lý toàn bộ dữ liệu và lưu vào `Data_Preprocessing`
Mỗi câu sau tiền xử lý được ghi trên 1 dòng.

In [ ]:
def preprocessing(text):
    text = text_std(text)              # 1. Chuẩn hóa văn bản
    text = to_lower(text)              # 2. Chữ thường
    text = remove_url(text)            # 3. Loại bỏ URL
    text = convert_emoji(text)         # 4. Emojis & Emoticons -> văn bản
    List_Sentences = split_sentences(text)   # 5. Tách câu
    List_Result = []
    for s in List_Sentences:
        s = remove_punctuation(s)      # 6. Xóa dấu câu
        s = remove_number(s)           # 7. Xóa chữ số
        s = correct_spelling(s)        # 8. Sửa lỗi chính tả
        s = tokenizing(s)              # 9. Tách từ
        s = remove_stopwords(s)        # 10. Xóa Stop Words
        if s != "":
            List_Result.append(s)
    return List_Result

for file_name in List_Files:
    f = open(os.path.join(path_in, file_name), "r", encoding="utf-8")
    text = f.read()
    f.close()

    List_Result = preprocessing(text)

    f = open(os.path.join(path_out, file_name), "w", encoding="utf-8")
    f.write("\n".join(List_Result))
    f.close()

print("Đã tiền xử lý và lưu", len(os.listdir(path_out)), "file vào", path_out)

## Kiểm tra kết quả

In [ ]:
# So sánh trước / sau tiền xử lý của 2 file đầu
for file_name in List_Files[:2]:
    print("=====", file_name, "=====")
    print("[Trước]:", open(os.path.join(path_in, file_name), "r", encoding="utf-8").read())
    print("[Sau]:")
    print(open(os.path.join(path_out, file_name), "r", encoding="utf-8").read())
    print()

# Kiểm tra còn sót dấu thanh tách rời không (kết quả mong muốn: 0)
count = 0
for file_name in os.listdir(path_out):
    text = open(os.path.join(path_out, file_name), "r", encoding="utf-8").read()
    count = count + len(re.findall(r"[\u0300-\u0323]", text))
print("Số dấu thanh còn tách rời:", count)